# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process data from the FAIR^2 dataset, using the [`mlcroissant`](https://mlcommons.github.io/croissant/python/main/index.html) library.

### Dataset Source
The dataset source is provided via a Croissant schema JSON-LD file URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access the metadata
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Croissant Schema Version: {metadata.conformsTo}")
print(f"License: {metadata.license}")

## 2. Data Overview

Review available record sets, fields, and their IDs. For referencing any entity in the dataset (record set, field, column, etc.), use their `@id` as required by the Croissant specification.

Let's list the record sets, fields, and columns provided by the dataset.

In [ ]:
# Retrieve all record sets
print("Record Sets (@id and name):")
record_sets = list(metadata.record_sets)
for rs in record_sets:
    print(f"  @id: {rs.id}, name: {rs.name}")

assert len(record_sets) > 0, "No record sets found in the dataset."

# Choose the first record set for further exploration
main_record_set = record_sets[0]
print(f"\nExploring record set: {main_record_set.id} (name: {main_record_set.name})\n")

print("Fields in the main record set:")
for field in main_record_set.fields:
    print(f"  @id: {field.id}\n    name: {field.name}\n    description: {field.description}\n    dataType: {field.data_type}\n")

## 3. Data Extraction

Load data from the main record set into a DataFrame for analysis. All manipulations will reference fields and columns by their Croissant `@id`.

In [ ]:
# Collect record set @ids
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for @id: {record_set_id}")
    try:
        records_iter = dataset.records(record_set=record_set_id)
        df = pd.DataFrame(list(records_iter))
        dataframes[record_set_id] = df
        print(f"  Loaded {len(df)} records\n")
    except Exception as e:
        print(f"  Failed to load records: {e}\n")

# Show available columns for the main record set
print(f"Columns for {main_record_set.id}:")
df_main = dataframes[main_record_set.id]
print(df_main.columns.tolist())
df_main.head()

## 4. Exploratory Data Analysis (EDA)

We analyze, filter, and transform the data using field `@id`s. As an example, we:
- Filter records based on a numeric field
- Normalize that field
- Group by a categorical field

Let's select a numeric and a group field (by their `@id`). The exact available fields will have been shown above.

In [ ]:
# Manually select a numeric field and a group field by @id
# These should be replaced as needed based on actual field IDs from previous output.

# For example, we use the following (replace with proper @id if different):
numeric_field_id = None
group_field_id = None

# Find likely numeric and group/categorical fields
for field in main_record_set.fields:
    if field.data_type and 'Float' in field.data_type:
        numeric_field_id = field.id
    if group_field_id is None and field.data_type and 'Text' in field.data_type:
        group_field_id = field.id

if numeric_field_id is None or numeric_field_id not in df_main.columns:
    # Fall back to searching by column types if not found
    for col in df_main.columns:
        # Try to guess which column is numeric
        if pd.api.types.is_numeric_dtype(df_main[col]):
            numeric_field_id = col
            break

if group_field_id is None or group_field_id not in df_main.columns:
    # Guess any non-numeric column as group_field
    for col in df_main.columns:
        if not pd.api.types.is_numeric_dtype(df_main[col]):
            group_field_id = col
            break

print(f"Numeric field selected (@id): {numeric_field_id}")
print(f"Group field selected  (@id): {group_field_id}\n")

# Proceed only if a numeric and group field are available
if numeric_field_id in df_main and group_field_id in df_main:
    # Filtering
    threshold = df_main[numeric_field_id].quantile(0.75) if pd.api.types.is_numeric_dtype(df_main[numeric_field_id]) else 10
    filtered_df = df_main[df_main[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df[[numeric_field_id, group_field_id]].head())

    # Normalization
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Grouping
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())
else:
    print("Cannot proceed with EDA: unable to find required numeric and group fields.")

## 5. Visualization

Let's visualize data distributions or relationships using `matplotlib`. We'll plot the distribution of the numeric field and the mean value per group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df_main and group_field_id in df_main:
    plt.figure(figsize=(8,4))
    sns.histplot(df_main[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Plot mean value per group
    grouped_df = df_main.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False).head(20)
    plt.figure(figsize=(10,5))
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
    plt.xticks(rotation=75, ha='right')
    plt.title(f"Top 20 {group_field_id} by mean {numeric_field_id}")
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to:
- Load dataset metadata and records using `mlcroissant`
- Explore dataset structure using Croissant `@id` references
- Perform simple exploratory analysis and data transformations
- Visualize numeric field distributions and group means

The FAIR^2 dataset on Mass Spectrometry provides rich protein-level records suitable for data-intensive proteomics explorations. For deeper analysis (e.g., machine learning), continue cleaning and transforming fields by their Croissant `@id`s to maintain domain interpretability.